# Heatmap ETHICS Regional — Chart 06–13

Panel disusun vertikal. Kanvas 36 inci dipakai untuk 17 persona dengan label regional yang panjang. Angka sel memakai 26 pt; angka biasa normal dan angka signifikan bertanda `*` bold. Output hanya PNG langsung ke `figures/`.

In [1]:
from __future__ import annotations

import hashlib
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

REPO_ROOT = Path(r'D:/TugasAkhir')
NOTEBOOK_DIR = REPO_ROOT / 'notebooks' / 'regional' / 'ethics accuracy analysis'
TABLE_DIR = NOTEBOOK_DIR / 'tables'
FIGURE_DIR = NOTEBOOK_DIR / 'figures'

# Change only these fixed values for a future visual revision.
FONT_SIZES = {
    'Judul utama': 30,
    'Judul panel': 26,
    'Label sumbu X/Y': 18,
    'Tick colorbar': 20,
    'Label colorbar': 23,
    'Angka sel': 26,
}

MODEL_LABELS = [
    'Qwen3 Next 80B-A3B', 'Llama 3.1 8B', 'Ministral 8B',
    'DeepSeek R1-Distill 32B', 'GPT-OSS 20B', 'Llama 3.3 70B',
]
PERSONA_LABELS = [
    'Asia Timur E', 'Asia Timur I1', 'Asia Timur I2', 'Asia Timur I3',
    'Amerika Utara E', 'Amerika Utara I1', 'Amerika Utara I2', 'Amerika Utara I3',
    'Asia Selatan E', 'Asia Selatan I1', 'Asia Selatan I2', 'Asia Selatan I3',
    'Afrika Sub-Sahara E', 'Afrika Sub-Sahara I1', 'Afrika Sub-Sahara I2', 'Afrika Sub-Sahara I3',
    'Neutral',
]
EFFECT_PERSONA_CODES = [
    'east_asian_explicit', 'east_asian_implicit_1', 'east_asian_implicit_2', 'east_asian_implicit_3',
    'north_american_explicit', 'north_american_implicit_1', 'north_american_implicit_2', 'north_american_implicit_3',
    'south_asian_explicit', 'south_asian_implicit_1', 'south_asian_implicit_2', 'south_asian_implicit_3',
    'sub_saharan_african_explicit', 'sub_saharan_african_implicit_1', 'sub_saharan_african_implicit_2', 'sub_saharan_african_implicit_3',
]
EFFECT_PERSONA_LABELS = [label for label in PERSONA_LABELS if label != 'Neutral']
SUBSET_LABELS = {'commonsense': 'Commonsense', 'deontology': 'Deontology', 'justice': 'Justice', 'virtue': 'Virtue'}
VALID_SUBSETS = ['commonsense', 'justice']
STANCE_SUBSETS = ['deontology', 'virtue']
SUBSET_ORDER = ['commonsense', 'deontology', 'justice', 'virtue']

summary = pd.read_csv(TABLE_DIR / 'ethics_condition_summary_source_derived.csv')
paired = pd.read_csv(TABLE_DIR / 'ethics_regional_parity_paired_effects_vs_neutral.csv')

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def summary_matrix(subset: str, metric: str) -> pd.DataFrame:
    frame = summary.loc[summary['subset'].eq(subset)]
    return frame.pivot(index='model_label', columns='persona_label', values=metric).reindex(index=MODEL_LABELS, columns=PERSONA_LABELS).mul(100)

def effect_matrices(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    values = frame.pivot(index='model_label', columns='persona', values='risk_difference_pp').reindex(index=MODEL_LABELS, columns=EFFECT_PERSONA_CODES)
    values.columns = EFFECT_PERSONA_LABELS
    labeled = frame.assign(annotation=frame.apply(lambda row: '' if pd.isna(row['risk_difference_pp']) else f"{row['risk_difference_pp']:.1f}" + ('*' if bool(row['significant_holm_0_05']) else ''), axis=1))
    annotations = labeled.pivot(index='model_label', columns='persona', values='annotation').reindex(index=MODEL_LABELS, columns=EFFECT_PERSONA_CODES)
    annotations.columns = EFFECT_PERSONA_LABELS
    return values, annotations

def style_axis(ax) -> None:
    ax.tick_params(axis='x', labelsize=FONT_SIZES['Label sumbu X/Y'], rotation=42, pad=5)
    ax.tick_params(axis='y', labelsize=FONT_SIZES['Label sumbu X/Y'], rotation=0, pad=5)
    for label in ax.get_xticklabels():
        label.set_horizontalalignment('right')
    for text in ax.texts:
        text.set_fontsize(FONT_SIZES['Angka sel'])
        text.set_fontweight('bold' if '*' in text.get_text() else 'normal')
    colorbar = ax.collections[0].colorbar
    colorbar.ax.tick_params(labelsize=FONT_SIZES['Tick colorbar'])
    colorbar.ax.yaxis.label.set_size(FONT_SIZES['Label colorbar'])

def save_png(fig, stem: str) -> Path:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    path = FIGURE_DIR / f'{stem}.png'
    fig.savefig(path, facecolor='#FAF9F7', bbox_inches='tight')
    plt.close(fig)
    return path

def plot_summary(metric: str, subsets: list[str], cmap: str, title: str, stem: str, colorbar_label: str) -> Path:
    fig, axes = plt.subplots(len(subsets), 1, figsize=(36, 8.7 * len(subsets)), squeeze=False, constrained_layout=True)
    for ax, subset in zip(axes.flat, subsets, strict=True):
        sns.heatmap(summary_matrix(subset, metric), annot=True, fmt='.1f', annot_kws={'fontsize': FONT_SIZES['Angka sel'], 'fontweight': 'normal'}, cmap=cmap, vmin=0, vmax=100, linewidths=0.5, cbar_kws={'label': colorbar_label}, ax=ax)
        ax.set_title(SUBSET_LABELS[subset], fontsize=FONT_SIZES['Judul panel'], fontweight='bold')
        ax.set_xlabel('')
        ax.set_ylabel('')
        style_axis(ax)
    fig.suptitle(title, fontsize=FONT_SIZES['Judul utama'], fontweight='bold')
    return save_png(fig, stem)

def plot_effects() -> list[Path]:
    specs = [
        ('coverage', 'Commit Rate', '10_ethics_regional_coverage_effect_holm_heatmap'),
        ('conditional_accuracy', 'Akurasi', '11_ethics_regional_conditional_accuracy_effect_holm_heatmap'),
        ('effective_success', 'Keberhasilan efektif', '12_ethics_regional_effective_success_effect_holm_heatmap'),
        ('stance_wrong', 'Distribusi keputusan', '13_ethics_regional_stance_wrong_effect_holm_heatmap'),
    ]
    paths = []
    for outcome, label, stem in specs:
        frame = paired.loc[paired['outcome'].eq(outcome)].copy()
        subsets = [subset for subset in SUBSET_ORDER if subset in set(frame['subset'])]
        limit = max(1.0, float(frame['risk_difference_pp'].abs().max()))
        fig, axes = plt.subplots(len(subsets), 1, figsize=(36, 8.7 * len(subsets)), squeeze=False, constrained_layout=True)
        for ax, subset in zip(axes.flat, subsets, strict=True):
            values, annotations = effect_matrices(frame.loc[frame['subset'].eq(subset)])
            sns.heatmap(values, annot=annotations, fmt='', annot_kws={'fontsize': FONT_SIZES['Angka sel'], 'fontweight': 'normal'}, cmap='RdBu_r', center=0, vmin=-limit, vmax=limit, linewidths=0.5, cbar_kws={'label': 'Risk difference (percentage points)'}, ax=ax)
            ax.set_title(SUBSET_LABELS[subset], fontsize=FONT_SIZES['Judul panel'], fontweight='bold')
            ax.set_xlabel('')
            ax.set_ylabel('')
            style_axis(ax)
        fig.suptitle(f'ETHICS Regional: {label}', fontsize=FONT_SIZES['Judul utama'], fontweight='bold')
        paths.append(save_png(fig, stem))
    return paths

sns.set_theme(style='white', font_scale=1)
plt.rcParams.update({'font.family': 'DejaVu Sans', 'figure.facecolor': '#FAF9F7', 'axes.facecolor': '#FAF9F7'})

rendered = [
    plot_summary('accuracy_committed', VALID_SUBSETS, 'RdYlGn', 'ETHICS Regional: Akurasi per Subset', '06_ethics_regional_accuracy_committed_valid_subsets_heatmap', 'Accuracy committed (%)'),
    plot_summary('effective_success_rate_nonmissing', VALID_SUBSETS, 'RdYlGn', 'ETHICS Regional: Keberhasilan Efektif per Subset', '07_ethics_regional_effective_success_valid_subsets_heatmap', 'Effective success (%)'),
    plot_summary('pct_answer_wrong', STANCE_SUBSETS, 'RdYlBu_r', 'ETHICS Regional: Distribusi Keputusan Deontology dan Virtue', '08_ethics_regional_stance_wrong_construct_invalid_heatmap', 'Stance wrong (%)'),
]
rendered.extend(plot_effects())

target_svg_names = {path.with_suffix('.svg').name for path in rendered}
for name in target_svg_names:
    svg_path = FIGURE_DIR / name
    if svg_path.exists():
        svg_path.unlink()

index_path = NOTEBOOK_DIR / 'output_index.csv'
index = pd.read_csv(index_path)
index = index.loc[~index['file'].isin(target_svg_names)].copy()
for path in rendered:
    matching_rows = index['file'].eq(path.name)
    if not matching_rows.any():
        index = pd.concat([index, pd.DataFrame([{'kind': 'figure', 'file': path.name, 'relative_path': f'figures/{path.name}', 'path': str(path), 'sha256': sha256_file(path)}])], ignore_index=True)
    else:
        index.loc[matching_rows, 'sha256'] = sha256_file(path)
index.to_csv(index_path, index=False)

active_path = NOTEBOOK_DIR / 'active_figures.csv'
active = pd.read_csv(active_path)
active = active.loc[~active['figure'].str.endswith(tuple(target_svg_names))].copy()
active.to_csv(active_path, index=False)

assert all(path.suffix == '.png' and path.exists() for path in rendered)
assert not any((FIGURE_DIR / name).exists() for name in target_svg_names)
assert all(index.loc[index['file'].eq(path.name), 'sha256'].iloc[0] == sha256_file(path) for path in rendered)
print('PNG langsung diperbarui:', len(rendered))


PNG langsung diperbarui: 7
